# MLflow Champion and Challenger Selection

This notebook evaluates all `READY` versions of the registered model `machine_failure_xgb`, validates their evaluation metadata and artifacts, ranks eligible candidates by validation PR-AUC, and assigns the two best versions to the MLflow aliases `champion` and `challenger`.

## Selection policy

- **Primary ranking metric:** `val_pr_auc` (Average Precision, if logged with scikit-learn's `average_precision_score`).
- **Additional eligibility metrics:** `val_precision`, `val_recall`, and `val_f1` must be present and valid.
- **Dataset comparability:** candidates must share the same non-empty `validation_dataset_id`.
- **Artifact validation:** every candidate model artifact must load successfully.
- **Aliases:** the highest-ranked eligible version becomes `@champion`; the second-highest becomes `@challenger`.
- **Test metrics are not used** for candidate ranking or promotion.

> Important: training runs must log the MLflow run tag `validation_dataset_id` with the same value whenever they use the same validation dataset. If this tag is absent or inconsistent, the notebook will stop before changing aliases. This prevents comparing validation scores produced on different validation datasets.


## Configuration

Update the values below to match your MLflow environment and validation policy.

`MIN_VALIDATION_PRECISION` and `MIN_VALIDATION_RECALL` are optional guardrails. Keep them as `None` until operational minimums have been agreed; set numeric values between `0.0` and `1.0` to enforce them.


In [ ]:
import os
import math
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

# MLflow connection
MLFLOW_TRACKING_URI = os.getenv(
    "MLFLOW_TRACKING_URI",
    "http://127.0.0.1:5000"
)

# Registered model and aliases
REGISTERED_MODEL_NAME = os.getenv(
    "MLFLOW_MODEL_NAME",
    "machine_failure_xgb"
)
CHAMPION_ALIAS = "champion"
CHALLENGER_ALIAS = "challenger"

# Validation metrics logged by the training runs
VALIDATION_METRICS = [
    "val_pr_auc",
    "val_precision",
    "val_recall",
    "val_f1",
]
RANKING_METRIC = "val_pr_auc"

# Dataset identity must be logged as an MLflow RUN TAG.
VALIDATION_DATASET_TAG = "validation_dataset_id"

# Optional operational guardrails.
# Example: 0.80 means candidates must have validation recall >= 0.80.
MIN_VALIDATION_PRECISION = None
MIN_VALIDATION_RECALL = None

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

for threshold_name, threshold_value in {
    "MIN_VALIDATION_PRECISION": MIN_VALIDATION_PRECISION,
    "MIN_VALIDATION_RECALL": MIN_VALIDATION_RECALL,
}.items():
    if threshold_value is not None and not 0.0 <= threshold_value <= 1.0:
        raise ValueError(f"{threshold_name} must be between 0.0 and 1.0.")

print(f"Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Registered model: {REGISTERED_MODEL_NAME}")
print(f"Ranking metric: {RANKING_METRIC}")
print(f"Validation dataset tag: {VALIDATION_DATASET_TAG}")


## Step 1 — Fetch candidate models

Retrieve every model version belonging to `machine_failure_xgb`. Only versions with status `READY` are considered in the following steps.


In [ ]:
try:
    client.get_registered_model(REGISTERED_MODEL_NAME)
except Exception as exc:
    raise RuntimeError(
        f"Registered model '{REGISTERED_MODEL_NAME}' was not found. "
        "Check the model name and MLflow tracking URI."
    ) from exc

all_versions = client.search_model_versions(
    filter_string=f"name = '{REGISTERED_MODEL_NAME}'"
)

ready_versions = [
    version
    for version in all_versions
    if str(version.status).upper() == "READY"
]

print(f"Total model versions found: {len(all_versions)}")
print(f"READY model versions: {len(ready_versions)}")

if not ready_versions:
    raise RuntimeError(
        f"No READY versions were found for '{REGISTERED_MODEL_NAME}'."
    )

versions_df = pd.DataFrame([
    {
        "version": int(version.version),
        "status": version.status,
        "run_id": version.run_id,
        "creation_time": pd.to_datetime(
            version.creation_timestamp,
            unit="ms",
            utc=True
        ),
    }
    for version in ready_versions
]).sort_values("version", ascending=False).reset_index(drop=True)

display(versions_df)


## Step 2 — Retrieve validation metrics

For each READY version, retrieve the four validation metrics from its source run:

- `val_pr_auc`
- `val_precision`
- `val_recall`
- `val_f1`

Also retrieve the `validation_dataset_id` run tag. This identifier is required to verify that candidate versions were evaluated on the same validation dataset.


In [ ]:
candidate_records = []

for version in sorted(ready_versions, key=lambda item: int(item.version)):
    record = {
        "version": int(version.version),
        "run_id": version.run_id,
        "val_pr_auc": None,
        "val_precision": None,
        "val_recall": None,
        "val_f1": None,
        "validation_dataset_id": None,
        "metric_error": None,
    }

    try:
        if not version.run_id:
            raise ValueError("The model version has no source run ID.")

        source_run = client.get_run(version.run_id)
        run_metrics = source_run.data.metrics
        run_tags = source_run.data.tags

        # Read metrics from the source training run.
        for metric_name in VALIDATION_METRICS:
            metric_value = run_metrics.get(metric_name)
            record[metric_name] = (
                float(metric_value) if metric_value is not None else None
            )

        # Read the validation dataset identifier from the source run.
        record["validation_dataset_id"] = run_tags.get(
            VALIDATION_DATASET_TAG
        )

    except Exception as exc:
        record["metric_error"] = str(exc)

    candidate_records.append(record)

metrics_df = pd.DataFrame(candidate_records)

display(
    metrics_df.sort_values(
        RANKING_METRIC,
        ascending=False,
        na_position="last"
    ).reset_index(drop=True)
)


## Step 3 — Candidate validation

A candidate is eligible only when all of the following conditions are met:

1. All four validation metrics are present, finite, and within the range `[0, 1]`.
2. The source run contains a non-empty `validation_dataset_id`.
3. The model artifact can be loaded successfully.
4. Optional minimum precision and recall guardrails are satisfied.

After individual checks, all eligible candidates must have the same `validation_dataset_id`. If dataset identity is missing or inconsistent, the notebook stops before modifying either alias.


In [ ]:
validation_results = []

for version in sorted(ready_versions, key=lambda item: int(item.version)):
    row = metrics_df.loc[
        metrics_df["version"] == int(version.version)
    ].iloc[0]

    reasons = []
    metric_values = {}

    # Check that all required metrics exist and are valid.
    for metric_name in VALIDATION_METRICS:
        value = row[metric_name]

        if pd.isna(value):
            reasons.append(f"Missing metric: {metric_name}")
        elif not math.isfinite(float(value)):
            reasons.append(f"Non-finite metric: {metric_name}")
        elif not 0.0 <= float(value) <= 1.0:
            reasons.append(f"Out-of-range metric: {metric_name}")
        else:
            metric_values[metric_name] = float(value)

    # Check validation dataset identity.
    dataset_id = row["validation_dataset_id"]
    if pd.isna(dataset_id) or not str(dataset_id).strip():
        reasons.append(
            f"Missing run tag: {VALIDATION_DATASET_TAG}"
        )

    # Check optional operational guardrails.
    if (
        MIN_VALIDATION_PRECISION is not None
        and "val_precision" in metric_values
        and metric_values["val_precision"] < MIN_VALIDATION_PRECISION
    ):
        reasons.append(
            f"Validation precision below minimum "
            f"({MIN_VALIDATION_PRECISION})"
        )

    if (
        MIN_VALIDATION_RECALL is not None
        and "val_recall" in metric_values
        and metric_values["val_recall"] < MIN_VALIDATION_RECALL
    ):
        reasons.append(
            f"Validation recall below minimum "
            f"({MIN_VALIDATION_RECALL})"
        )

    # Load the exact registered model version to verify its artifact.
    artifact_loaded = False
    if not reasons:
        try:
            model_uri = (
                f"models:/{REGISTERED_MODEL_NAME}/{version.version}"
            )
            loaded_model = mlflow.pyfunc.load_model(model_uri)
            artifact_loaded = loaded_model is not None
            del loaded_model
        except Exception as exc:
            reasons.append(f"Artifact loading failed: {exc}")

    validation_results.append({
        "version": int(version.version),
        "run_id": version.run_id,
        "validation_dataset_id": dataset_id,
        **metric_values,
        "artifact_loaded": artifact_loaded,
        "eligible": len(reasons) == 0,
        "rejection_reasons": "; ".join(reasons),
    })

validation_df = pd.DataFrame(validation_results)

eligible_df = validation_df[
    validation_df["eligible"]
].copy()

if eligible_df.empty:
    display(validation_df)
    raise RuntimeError(
        "No eligible model candidates remain. Review the validation "
        "metrics, dataset tag, guardrails, and model artifacts."
    )

# All eligible candidates must use the same validation dataset.
dataset_ids = (
    eligible_df["validation_dataset_id"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

if len(dataset_ids) != 1:
    display(validation_df)
    raise RuntimeError(
        "Validation dataset mismatch: eligible candidates do not share "
        "one validation_dataset_id. No aliases have been changed."
    )

print(f"Validated dataset ID: {dataset_ids[0]}")
print(f"Eligible candidates: {len(eligible_df)}")

display(validation_df)


## Step 4 — Rank candidates

Rank eligible candidates by `val_pr_auc` in descending order. Validation recall and precision act as eligibility guardrails when their minimum thresholds are configured.

If two candidates have the same validation PR-AUC, the higher model version number is used as the tie-breaker. The first-ranked candidate is assigned to `@champion`; the second-ranked candidate is assigned to `@challenger`.


In [ ]:
ranked_candidates = (
    eligible_df
    .sort_values(
        by=["val_pr_auc", "version"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranked_candidates.insert(
    0,
    "rank",
    range(1, len(ranked_candidates) + 1)
)

ranked_candidates["proposed_alias"] = None
ranked_candidates.loc[0, "proposed_alias"] = CHAMPION_ALIAS

if len(ranked_candidates) >= 2:
    ranked_candidates.loc[1, "proposed_alias"] = CHALLENGER_ALIAS

display(
    ranked_candidates[
        [
            "rank",
            "version",
            "val_pr_auc",
            "val_precision",
            "val_recall",
            "val_f1",
            "validation_dataset_id",
            "proposed_alias",
        ]
    ]
)

champion_candidate = ranked_candidates.iloc[0]
challenger_candidate = (
    ranked_candidates.iloc[1]
    if len(ranked_candidates) >= 2
    else None
)

print(
    f"Proposed champion: version {int(champion_candidate['version'])} "
    f"(val_pr_auc={champion_candidate['val_pr_auc']:.6f})"
)

if challenger_candidate is not None:
    print(
        f"Proposed challenger: version {int(challenger_candidate['version'])} "
        f"(val_pr_auc={challenger_candidate['val_pr_auc']:.6f})"
    )
else:
    print("No challenger candidate is available.")


## Step 5 — Promote champion and challenger

Assign the highest-ranked eligible version to the `champion` alias and the second-highest eligible version to the `challenger` alias.

If only one eligible candidate exists, the notebook removes any existing `challenger` alias so that an outdated challenger is not left behind. The notebook then reads both aliases back from MLflow to verify the final assignments.


In [ ]:
# Promote the highest-ranked candidate to @champion.
champion_version = str(int(champion_candidate["version"]))

client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias=CHAMPION_ALIAS,
    version=champion_version,
)

# Promote the second-ranked candidate to @challenger.
if challenger_candidate is not None:
    challenger_version = str(int(challenger_candidate["version"]))

    client.set_registered_model_alias(
        name=REGISTERED_MODEL_NAME,
        alias=CHALLENGER_ALIAS,
        version=challenger_version,
    )
else:
    challenger_version = None

    # Remove a stale challenger alias when no second candidate exists.
    try:
        client.delete_registered_model_alias(
            name=REGISTERED_MODEL_NAME,
            alias=CHALLENGER_ALIAS,
        )
        print("Removed the existing challenger alias: no second candidate.")
    except Exception:
        print("No existing challenger alias needed to be removed.")

# Verify the aliases after promotion.
champion_alias_version = client.get_model_version_by_alias(
    name=REGISTERED_MODEL_NAME,
    alias=CHAMPION_ALIAS,
)

assert str(champion_alias_version.version) == champion_version, (
    "Champion alias verification failed."
)

if challenger_version is not None:
    challenger_alias_version = client.get_model_version_by_alias(
        name=REGISTERED_MODEL_NAME,
        alias=CHALLENGER_ALIAS,
    )

    assert str(challenger_alias_version.version) == challenger_version, (
        "Challenger alias verification failed."
)

print("Alias promotion completed and verified.")
print(
    f"Champion URI: "
    f"models:/{REGISTERED_MODEL_NAME}@{CHAMPION_ALIAS}"
)
if challenger_version is not None:
    print(
        f"Challenger URI: "
        f"models:/{REGISTERED_MODEL_NAME}@{CHALLENGER_ALIAS}"
    )
else:
    print("Challenger URI: not assigned (only one eligible candidate).")

promotion_summary = pd.DataFrame([
    {
        "alias": CHAMPION_ALIAS,
        "version": champion_version,
        "val_pr_auc": float(champion_candidate["val_pr_auc"]),
        "run_id": champion_candidate["run_id"],
    },
    {
        "alias": CHALLENGER_ALIAS,
        "version": challenger_version,
        "val_pr_auc": (
            float(challenger_candidate["val_pr_auc"])
            if challenger_candidate is not None
            else None
        ),
        "run_id": (
            challenger_candidate["run_id"]
            if challenger_candidate is not None
            else None
        ),
    },
])

display(promotion_summary)


## Operational notes

- The notebook uses **validation metrics only** for candidate selection. Test metrics should remain reserved for final, unbiased model evaluation.
- `val_pr_auc` should be logged consistently across training runs. If it comes from scikit-learn's `average_precision_score`, it is Average Precision (AP), not necessarily the trapezoidal area under a precision-recall curve.
- The `validation_dataset_id` tag must identify the exact validation dataset and preprocessing/split definition. Use the same identifier only when those evaluation inputs are genuinely comparable.
- If the validation dataset changes, assign a new identifier and evaluate candidates intended for comparison on that same dataset.
- Configure minimum validation precision and recall only after the operational trade-off between false positives and false negatives has been agreed.
- The aliases are mutable pointers. Applications can load the selected versions through `models:/machine_failure_xgb@champion` and `models:/machine_failure_xgb@challenger`.
- This notebook does not delete model versions or automatically deploy a model to the inference service. It only updates and verifies MLflow aliases.
